# Challenge 03 — Connect Your Agents to Tools

In this challenge, you'll connect your **NewsAgent** to tools using the Azure AI Agent SDK. By the end, your agent will be able to:
- Call external functions (function calling)
- Search through uploaded files (file search)
- Connect to Azure AI Search for production-grade RAG

## Learning Objectives
- Define function tool schemas and attach them to an agent
- Handle the function calling loop (detect `requires_action`, provide tool output)
- Upload files and create vector stores for File Search
- Connect an agent to Azure AI Search
- Observe grounded vs. hallucinated responses

---
## Setup — Environment & Authentication

This section is filled out for you. It loads your `.env` file and connects to your Foundry project — the same pattern you used in Challenge 02.

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential, InteractiveBrowserCredential, ChainedTokenCredential
from azure.ai.agents import AgentsClient
from azure.ai.agents.models import (
    MessageRole,
    FunctionTool,
    ToolSet,
    FilePurpose,
    FileSearchTool,
    VectorStore,
)
import json

# Load environment variables from .env
load_dotenv()

# These are pre-filled from your .env — same values you used in Challenge 02
PROJECT_ENDPOINT = os.getenv("PROJECT_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("MODEL_DEPLOYMENT_NAME")
TENANT_ID = os.getenv("AZURE_TENANT_ID")
AZURE_SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_INDEX_NAME = os.getenv("AZURE_SEARCH_INDEX_NAME")

# Validate required values are present
assert PROJECT_ENDPOINT, "PROJECT_ENDPOINT is missing from .env"
assert MODEL_DEPLOYMENT_NAME, "MODEL_DEPLOYMENT_NAME is missing from .env"
assert AZURE_SEARCH_ENDPOINT, "AZURE_SEARCH_ENDPOINT is missing from .env"
assert AZURE_SEARCH_INDEX_NAME, "AZURE_SEARCH_INDEX_NAME is missing from .env"

print(f"✅ Endpoint: {PROJECT_ENDPOINT}")
print(f"✅ Model: {MODEL_DEPLOYMENT_NAME}")
print(f"✅ Search: {AZURE_SEARCH_ENDPOINT}")
print(f"✅ Index: {AZURE_SEARCH_INDEX_NAME}")

In [ ]:
# Authenticate — same pattern as Challenge 02
credential = ChainedTokenCredential(
    AzureCliCredential(tenant_id=TENANT_ID) if TENANT_ID else AzureCliCredential(),
    InteractiveBrowserCredential(tenant_id=TENANT_ID) if TENANT_ID else InteractiveBrowserCredential(),
)

# Create the AgentsClient
agents_client = AgentsClient(
    endpoint=PROJECT_ENDPOINT,
    credential=credential,
)
print("✅ Connected to Azure AI Foundry")

---
## Part 1: Function Calling — Define a Tool Schema

Function calling lets your agent request external data by generating structured JSON calls. You define the function schema (name, description, parameters), and when the agent decides it needs that data, it emits a function call for your code to handle.

**Your task:** Define a function tool schema for `get_news_articles` that the agent can call to retrieve news.

In [ ]:
# TODO: Define the function schema for get_news_articles
# The agent will call this function when it needs news data.
#
# Your function should accept these parameters:
#   - topic (string, required): The news topic to search for
#   - region (string, optional): Geographic region to filter results
#   - max_results (integer, optional): Number of articles to return (default 3)
#
# Hint: Use the FunctionTool class and define parameters as a JSON schema dict

get_news_articles_schema = {
    "type": "function",
    "function": {
        "name": "get_news_articles",
        "description": "___",  # TODO: Write a clear description of what this function does
        "parameters": {
            "type": "object",
            "properties": {
                "topic": {
                    "type": "string",
                    "description": "___"  # TODO: Describe this parameter
                },
                "region": {
                    "type": "___",  # TODO: What type is this?
                    "description": "Geographic region to filter news results (e.g., 'Europe', 'Asia')"
                },
                "max_results": {
                    "type": "___",  # TODO: What type is this?
                    "description": "___"  # TODO: Describe this parameter
                }
            },
            "required": ["___"]  # TODO: Which parameters are required?
        }
    }
}

# Create the FunctionTool from your schema
functions = FunctionTool(definitions=[get_news_articles_schema])
print("✅ Function tool schema defined")
print(json.dumps(get_news_articles_schema, indent=2))

---
## Part 2: Create the Agent with Function Calling

Now create a NewsAgent that has the function tool attached. When this agent needs news data, it will emit a `get_news_articles` function call instead of making things up.

**Your task:** Create the agent with appropriate system instructions and the function tool.

In [ ]:
# TODO: Create the agent with the function tool attached
#
# Requirements:
#   - Name: "NewsAgentWithTools"
#   - Model: use MODEL_DEPLOYMENT_NAME
#   - Instructions: Tell the agent it is a travel news assistant that should:
#       * Always use the get_news_articles tool to retrieve current news
#       * Never make up news articles or events
#       * Summarize results in a clear, organized format
#   - Tools: attach the function tool you defined above
#
# Hint: Use agents_client.create_agent() with a toolset parameter

toolset = ToolSet()
toolset.definitions.append(functions)

news_agent_with_tools = agents_client.create_agent(
    model=___,  # TODO: Which model deployment?
    name="___",  # TODO: What name?
    instructions=(
        "___"  # TODO: Write system instructions for the agent
        # Remember: Tell the agent to USE the get_news_articles tool
        # and to never fabricate news articles
    ),
    toolset=toolset,
)
print(f"✅ Agent created — ID: {news_agent_with_tools.id}")

---
## Part 3: Handle the Function Calling Loop

When you send a message to an agent with function tools, the agent may respond with a `requires_action` status instead of a completed message. This means the agent wants to call your function. You must:
1. Detect the `requires_action` status
2. Extract the function name and arguments
3. Execute the function (or return simulated data)
4. Submit the tool output back to the agent
5. Let the agent generate its final response

**Your task:** Implement the function calling loop. The simulated news data is provided for you — focus on the loop logic.

In [ ]:
# Simulated news data — in production this would call a real API
# This is provided for you; no changes needed here.
def get_news_articles(topic: str, region: str = None, max_results: int = 3) -> str:
    """Simulated news retrieval function."""
    articles = [
        {
            "headline": f"Major {topic} developments reported in {region or 'global'} markets",
            "summary": f"Experts highlight significant changes in {topic} affecting travelers and businesses alike.",
            "source": "Travel Wire",
            "date": "2026-05-25"
        },
        {
            "headline": f"New {topic} regulations announced for international travelers",
            "summary": f"Government officials released updated guidelines regarding {topic} that may impact travel plans.",
            "source": "Global News Daily",
            "date": "2026-05-24"
        },
        {
            "headline": f"{region or 'World'} summit addresses {topic} concerns",
            "summary": f"Leaders from multiple countries convened to discuss {topic} and its implications for tourism.",
            "source": "International Herald",
            "date": "2026-05-23"
        },
    ]
    return json.dumps(articles[:max_results])

print("✅ Simulated news function ready")
print("Sample output:")
print(get_news_articles("travel safety", "Europe", 2))

In [ ]:
# Create a thread and send a message that should trigger function calling
thread = agents_client.threads.create()
print(f"✅ Thread created — ID: {thread.id}")

# Send a message that requires the agent to look up news
user_message = "What are the latest travel advisories for Europe?"
agents_client.messages.create(
    thread_id=thread.id,
    role=MessageRole.USER,
    content=user_message,
)
print(f"✅ Message sent: '{user_message}'")

In [ ]:
# TODO: Implement the function calling loop
#
# Steps:
# 1. Create a run with agents_client.runs.create()
# 2. Poll/wait for the run status
# 3. If status is "requires_action":
#    a. Extract tool_calls from run.required_action.submit_tool_outputs.tool_calls
#    b. For each tool call, get the function name and arguments
#    c. Call your local get_news_articles() function with those arguments
#    d. Submit the result back using agents_client.runs.submit_tool_outputs()
# 4. Wait for the run to complete
# 5. Read and print the agent's final response
#
# Hint: tool_call.function.name gives you the function name
# Hint: tool_call.function.arguments is a JSON string you need to parse
# Hint: tool_call.id is needed when submitting tool outputs

import time

# Step 1: Create the run
run = agents_client.runs.create(
    thread_id=thread.id,
    agent_id=___,  # TODO: Which agent ID?
)
print(f"Run created — ID: {run.id}, Status: {run.status}")

# Step 2: Poll for completion or requires_action
while run.status in ["queued", "in_progress"]:
    time.sleep(1)
    run = agents_client.runs.get(thread_id=thread.id, run_id=run.id)
    print(f"  Status: {run.status}")

# Step 3: Handle function calling if required
if run.status == "requires_action":
    print("\n🔧 Agent wants to call a function!")
    
    tool_calls = ___  # TODO: Extract tool_calls from run.required_action
    tool_outputs = []
    
    for tool_call in tool_calls:
        function_name = ___  # TODO: Get the function name from the tool_call
        function_args = json.loads(___)  # TODO: Get and parse the function arguments
        
        print(f"  Function: {function_name}")
        print(f"  Arguments: {function_args}")
        
        # Call our simulated function
        if function_name == "get_news_articles":
            result = get_news_articles(**function_args)
        else:
            result = json.dumps({"error": f"Unknown function: {function_name}"})
        
        tool_outputs.append({
            "tool_call_id": ___,  # TODO: What ID does the tool output need?
            "output": result,
        })
    
    # Step 4: Submit tool outputs back to the agent
    agents_client.runs.submit_tool_outputs(
        thread_id=___,  # TODO: Which thread?
        run_id=___,  # TODO: Which run?
        tool_outputs=tool_outputs,
    )
    print("\n✅ Tool outputs submitted")
    
    # Wait for the agent to finish processing
    while run.status in ["queued", "in_progress"]:
        time.sleep(1)
        run = agents_client.runs.get(thread_id=thread.id, run_id=run.id)

# Step 5: Print the final response
print(f"\n--- Final run status: {run.status} ---")
messages = agents_client.messages.list(thread_id=thread.id)
for msg in messages:
    role = msg.role
    text = msg.content[0].text.value if msg.content else ""
    print(f"\n{'='*50}")
    print(f"{role.upper()}:")
    print(f"{text}")

---
## Part 4: File Search — Upload Files and Create a Vector Store

File Search lets your agent search through uploaded documents (PDFs, spreadsheets, text files) using semantic search. You upload files, create a vector store, and attach it to your agent.

**Your task:** Upload the hotel data file and create an agent that can search through it.

In [ ]:
# TODO: Upload a file and create a vector store for the agent
#
# Steps:
# 1. Upload a file using agents_client.files.upload()
# 2. Create a vector store using agents_client.vector_stores.create()
# 3. Add the file to the vector store
# 4. Create an agent with FileSearchTool attached
#
# The hotel data file path is provided for you.
# Hint: FilePurpose.AGENTS is the purpose for agent-accessible files
# Hint: FileSearchTool(vector_store_ids=[...]) attaches the store to the agent

# Path to your hotel data (update if your file is in a different location)
HOTEL_DATA_FILE = "./Challenge-03/hotels.xlsx"  # Update this path as needed

# Step 1: Upload the file
uploaded_file = agents_client.files.upload(
    file_path=___,  # TODO: Which file to upload?
    purpose=___,  # TODO: What purpose? (Hint: FilePurpose.AGENTS)
)
print(f"✅ File uploaded — ID: {uploaded_file.id}")

# Step 2: Create a vector store
vector_store = agents_client.vector_stores.create(
    name="___",  # TODO: Give your vector store a descriptive name
    file_ids=[___],  # TODO: Which file ID(s) to include?
)
print(f"✅ Vector store created — ID: {vector_store.id}")

In [ ]:
# TODO: Create an agent with File Search enabled
#
# This agent should be able to search through the hotel data
# and answer questions about specific hotels, prices, and amenities.

file_search_tool = FileSearchTool(vector_store_ids=[___])  # TODO: Which vector store?

travel_agent_with_search = agents_client.create_agent(
    model=MODEL_DEPLOYMENT_NAME,
    name="___",  # TODO: Give the agent a name
    instructions=(
        "___"  # TODO: Write instructions telling the agent to:
        # - Search through uploaded hotel/flight data to answer questions
        # - Always cite specific data (prices, names, amenities) from files
        # - Never make up hotel names or prices
        # - If information is not in the files, say so clearly
    ),
    tools=file_search_tool.definitions,
    tool_resources=file_search_tool.resources,
)
print(f"✅ Agent with File Search created — ID: {travel_agent_with_search.id}")

In [ ]:
# Test the File Search agent — this is filled out for you
thread_fs = agents_client.threads.create()

# Ask a question that requires searching the uploaded file
test_query = "What hotels are available and what are their nightly rates?"
agents_client.messages.create(
    thread_id=thread_fs.id,
    role=MessageRole.USER,
    content=test_query,
)

# Run and wait
run_fs = agents_client.runs.create_and_process(
    thread_id=thread_fs.id,
    agent_id=travel_agent_with_search.id,
)
print(f"Run status: {run_fs.status}")

# Print response
messages_fs = agents_client.messages.list(thread_id=thread_fs.id)
for msg in messages_fs:
    role = msg.role
    text = msg.content[0].text.value if msg.content else ""
    print(f"\n--- {role.upper()} ---\n{text}")

---
## Part 5: Azure AI Search — Production-Grade RAG

For production applications with large document collections, you connect your agent to Azure AI Search instead of a simple vector store. This gives you enterprise-scale retrieval with advanced features like semantic ranking, filters, and hybrid search.

**Your task:** Create an agent connected to your Azure AI Search index.

In [ ]:
from azure.ai.agents.models import AzureAISearchTool, ConnectionProperties, ConnectionType

# TODO: Configure the Azure AI Search tool connection
#
# You need to create a search tool that points to your AI Search index.
# The endpoint and index name are already loaded from .env above.
#
# Hint: AzureAISearchTool needs:
#   - index_name: the name of your search index
#   - index_connection: a ConnectionProperties object with your search endpoint

ai_search_tool = AzureAISearchTool(
    index_name=___,  # TODO: Which index? (already loaded from .env)
    index_connection=ConnectionProperties(
        connection_type=ConnectionType.AZURE_AI_SEARCH,
        endpoint=___,  # TODO: Which endpoint? (already loaded from .env)
    ),
)

print(f"✅ AI Search tool configured for index: {AZURE_SEARCH_INDEX_NAME}")

In [ ]:
# TODO: Create an agent with the Azure AI Search tool
#
# This agent should use AI Search to find relevant news and events
# from your indexed documents.

news_agent_with_search = agents_client.create_agent(
    model=___,  # TODO: Which model?
    name="___",  # TODO: Agent name (e.g., "NewsAgentWithSearch")
    instructions=(
        "___"  # TODO: Write instructions telling the agent to:
        # - Use Azure AI Search to find relevant news and events
        # - Always ground responses in search results
        # - Cite sources when providing information
        # - If no relevant results found, clearly state that
    ),
    tools=ai_search_tool.definitions,
    tool_resources=ai_search_tool.resources,
)
print(f"✅ Agent with AI Search created — ID: {news_agent_with_search.id}")

In [ ]:
# Test the AI Search agent — this is filled out for you
thread_search = agents_client.threads.create()

# Ask something that requires searching your index
search_query = "Are there any upcoming music festivals in Barcelona?"
agents_client.messages.create(
    thread_id=thread_search.id,
    role=MessageRole.USER,
    content=search_query,
)

# Run and wait
run_search = agents_client.runs.create_and_process(
    thread_id=thread_search.id,
    agent_id=news_agent_with_search.id,
)
print(f"Run status: {run_search.status}")

# Print response
messages_search = agents_client.messages.list(thread_id=thread_search.id)
for msg in messages_search:
    role = msg.role
    text = msg.content[0].text.value if msg.content else ""
    print(f"\n--- {role.upper()} ---\n{text}")

---
## 🧪 Verification — Test Grounding vs. Hallucination

Run the cell below to compare an agent WITH tools vs WITHOUT tools on the same question. This demonstrates why grounding matters.

In [ ]:
# Compare grounded vs. ungrounded responses
# This cell is complete — just run it and observe the difference

test_question = "What is the nightly rate for the cheapest hotel available?"

# --- Agent WITHOUT tools (will hallucinate) ---
bare_agent = agents_client.create_agent(
    model=MODEL_DEPLOYMENT_NAME,
    name="BareAgent-Test",
    instructions="You are a travel assistant. Answer questions about hotels.",
)

thread_bare = agents_client.threads.create()
agents_client.messages.create(thread_id=thread_bare.id, role=MessageRole.USER, content=test_question)
run_bare = agents_client.runs.create_and_process(thread_id=thread_bare.id, agent_id=bare_agent.id)
messages_bare = agents_client.messages.list(thread_id=thread_bare.id)

print("=" * 60)
print("❌ AGENT WITHOUT TOOLS (likely hallucinated):")
print("=" * 60)
for msg in messages_bare:
    if msg.role == MessageRole.AGENT:
        print(msg.content[0].text.value if msg.content else "")

# --- Agent WITH File Search (grounded) ---
thread_grounded = agents_client.threads.create()
agents_client.messages.create(thread_id=thread_grounded.id, role=MessageRole.USER, content=test_question)
run_grounded = agents_client.runs.create_and_process(thread_id=thread_grounded.id, agent_id=travel_agent_with_search.id)
messages_grounded = agents_client.messages.list(thread_id=thread_grounded.id)

print("\n")
print("=" * 60)
print("✅ AGENT WITH FILE SEARCH (grounded in data):")
print("=" * 60)
for msg in messages_grounded:
    if msg.role == MessageRole.AGENT:
        print(msg.content[0].text.value if msg.content else "")

# Cleanup test agent
agents_client.delete_agent(bare_agent.id)
print("\n\n💡 Notice the difference? The grounded agent cites real data.")
print("   The ungrounded agent makes up plausible-sounding prices.")

---
## Cleanup

⚠️ **Do NOT delete your agents yet** — you will need them in Challenge 04 (MCP) and Challenge 05 (Orchestration).

Only run the cleanup cell below if you need to start over.

In [ ]:
# ONLY run this if you need to start over — your agents are needed for later challenges!
# Uncomment the lines below to delete:

# agents_client.delete_agent(news_agent_with_tools.id)
# print(f"Deleted NewsAgentWithTools — ID: {news_agent_with_tools.id}")

# agents_client.delete_agent(travel_agent_with_search.id)
# print(f"Deleted TravelAgentWithSearch — ID: {travel_agent_with_search.id}")

# agents_client.delete_agent(news_agent_with_search.id)
# print(f"Deleted NewsAgentWithSearch — ID: {news_agent_with_search.id}")

print("ℹ️ Agents preserved for Challenge 04 and beyond.")
print(f"  NewsAgentWithTools ID: {news_agent_with_tools.id}")
print(f"  TravelAgentWithSearch ID: {travel_agent_with_search.id}")
print(f"  NewsAgentWithSearch ID: {news_agent_with_search.id}")